In [ ]:
# Mount Google Drive for import and export
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy Keras Tuner Values
!cp -r "/content/drive/My Drive/Colab Notebooks/cifar_tuning/cifar_tuning" "./"

In [ ]:
# Copy Cleaned Dataset from Google Drive
!cp "/content/drive/MyDrive/Colab Notebooks/cifar10_cleaned.npz" "/content/cifar10_cleaned.npz"


In [ ]:
# This is just for installing keras tuner for hyperparameter optimization
!pip install keras-tuner

In [ ]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Global Parameters
BATCH_SIZE = 128 # Using Google Colab Notebook with T4 GPU, so it is more than capable of using this size.
EPOCHS = 200 # Max Epochs, but early stopping is configured for actual training
NUM_CLASSES = 10 # Number of classes in CIFAR Dataset
INPUT_SHAPE = (32, 32, 3)
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/CIFAR-10-C" # Location of CIFAR-10-C in Google Colab
CLASS_NAMES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

In [ ]:
# Function to load prepared data from data cleaning pipeline
def load_prepared_data(filepath='cifar10_cleaned.npz'):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File {filepath} not found.")

    with np.load(filepath) as data:
        x_train = data['x_train']
        y_train = data['y_train']
        x_test = data['x_test']
        y_test = data['y_test']
        x_val = data['x_val']
        y_val = data['y_val']
    print("Data loaded successfully.")
    print("x_train shape:", x_train.shape)
    print("y_train shape:", y_train.shape)
    print("x_test shape:", x_test.shape)
    print("y_test shape:", y_test.shape)
    print("x_val shape:", x_val.shape)
    print("y_val shape:", y_val.shape)
    return x_train, y_train, x_test, y_test, x_val, y_val

In [ ]:
# Import prepared data from cleaning algorithm
x_train, y_train, x_test, y_test, x_val, y_val = load_prepared_data()

In [ ]:
# Function Defines model. Use_robust allows the heavy augmentation layers to be removed to
# establish baseline and robust models separately.
def build_model(hp, use_robust = True):
    model = models.Sequential()
    model.add(layers.Input(shape=INPUT_SHAPE))

    if use_robust:
        model.add(layers.RandomContrast(0.2))
        model.add(layers.RandomBrightness(0.05))
        model.add(layers.RandomRotation(0.05))
        model.add(layers.RandomZoom(0.1))
        model.add(layers.RandomColorDegeneration(0.02))
        model.add(layers.GaussianNoise(0.05))

    model.add(layers.RandomFlip(mode='horizontal'))

    model.add(layers.RandomTranslation(0.1, 0.1))



    # Block 1 (Edges)
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Hyperparameter: Dropout
    dropout_rate = hp.Float("dropout", 0.0, 0.5, step=0.1)
    model.add(layers.Dropout(dropout_rate))

    # Block 2 (Shapes)
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(dropout_rate))

    # Block 3 (Objects)
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(dropout_rate))

    # Head
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # Hyperparameter: Learning Rate
    lr = hp.Float("learning_rate", 0.00001, 0.001, sampling="log")

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=['accuracy'])

    return model






In [ ]:
# Configure Baseline and Robust model keras tuner parameters.
tuner_baseline = kt.BayesianOptimization(
    hypermodel=lambda hp: build_model(hp, use_robust = False),
    objective='val_accuracy',
    max_trials=24,
    directory='cifar_tuning',
    project_name='baysean_opt_baseline_v1.0',
    num_initial_points=6,
    overwrite=False,
    alpha=0.001,
    beta=5.0,
    seed=42
)
tuner_robust = kt.BayesianOptimization(
    hypermodel=lambda hp: build_model(hp, use_robust = True),
    objective='val_accuracy',
    max_trials=24,
    directory='cifar_tuning',
    project_name='baysean_opt_robust_v2.0',
    num_initial_points=6,
    overwrite=False,
    alpha=0.001,
    beta=5.0,
    seed=42
)

In [ ]:
# Configure Early Stopping for parameter search.
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
# Perform Bayesean Optimization of parameters to find best LR and dropout parameters on Robust model
tuner_baseline.search(
    x_train, y_train,
    epochs=30,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=2
)

best_hps = tuner_baseline.get_best_hyperparameters()[0]
print(f"Best Baseline Learning Rate: {best_hps.get('learning_rate')}")
print(f"Best Baseline Dropout Rate: {best_hps.get('dropout')}")

In [ ]:
# Perform Bayesean Optimization of parameters to find best LR and dropout parameters on Robust model
tuner_robust.search(
    x_train, y_train,
    epochs=30,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=2
)

best_hps_robust = tuner_robust.get_best_hyperparameters()[0]
print(f"Best Robust Learning Rate: {best_hps_robust.get('learning_rate')}")
print(f"Best Robust Dropout Rate: {best_hps_robust.get('dropout')}")

In [ ]:
# Copy Keras Tuner Data to Google Drive
!cp -r "/content/cifar_tuning" "/content/drive/MyDrive/Colab Notebooks/cifar_tuning"

In [ ]:
# Configure callbacks to include dynamic reduction of LR on stagnation and early stopping.
callbacks_list = [
    callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=7, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1)
]

In [ ]:
# Train Baseline Model
print("STARTING BASELINE TRAINING...")
model_baseline = build_model(best_hps, use_robust=False)
history_baseline = model_baseline.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=callbacks_list,
    verbose=2
)
model_baseline.save('final_baseline.keras')

In [ ]:
# Train Robust Model
print("STARTING ROBUST TRAINING...")
model_robust = build_model(best_hps_robust, use_robust=True)
history_robust = model_robust.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=callbacks_list,
    verbose=2
)
model_robust.save('final_robust.keras')

In [ ]:
model_baseline.summary()

In [ ]:
model_robust.summary()

In [ ]:
# Perform predictions and output predictions shape.
y_preds_robust = model_robust.predict(x_test)
print(f"Robust predictions shape: {y_preds_robust.shape}")
y_preds_baseline = model_baseline.predict(x_test)
print(f"Baseline predictions shape: {y_preds_baseline.shape}")




In [ ]:
# Evaluate both models against first the clean dataset in CIFAR-10 and then the corrupted datasets of CIFAR-10-C.
corruptions = [
    'brightness.npy',
    'gaussian_noise.npy',
    'shot_noise.npy',
    'impulse_noise.npy',
    'defocus_blur.npy',
    'glass_blur.npy',
    'motion_blur.npy',
    'zoom_blur.npy',
    'snow.npy',
    'frost.npy',
    'fog.npy',
    'contrast.npy',
    'elastic_transform.npy',
    'pixelate.npy',
    'jpeg_compression.npy',
    'pixelate.npy',
    'speckle_noise.npy',
    'gaussian_blur.npy',
    'spatter.npy',
]

# Load Labels (Common to all corruptions)
# Try to find labels.npy
label_path = os.path.join(DATA_PATH, 'labels.npy')

if os.path.exists(label_path):
    labels = np.load(label_path)
    # Convert to one-hot (CIFAR-10 has 10 classes)
    y_test_corrupt = tf.keras.utils.to_categorical(labels, 10)
    print("Labels loaded successfully.")
else:
    print(f"ERROR: Could not find labels.npy at {label_path}")
    print("Please check your path!")
    # Stop execution if no labels
    raise FileNotFoundError("labels.npy missing")

# --- 2. Load Both Models
print("Loading models from disk...")
try:
    model_baseline = tf.keras.models.load_model('final_baseline.keras')
    print("Baseline loaded.")
    model_robust = tf.keras.models.load_model('final_robust.keras')
    print("Robust loaded.")
except Exception as e:
    print(f"Error loading models: {e}")
    raise e

# --- 3. Test Corruptions ---
print("="*72)
print(f"{'CORRUPTION':<20} | {'BASE ACC':<10} | {'ROBUST ACC':<10} | {'DIFFERENCE'}")
print("="*72)
loss_b, acc_b = model_baseline.evaluate(x_test, y_test, batch_size=128, verbose=0)
loss_r, acc_r = model_robust.evaluate(x_test, y_test, batch_size=128, verbose=0)
diff = acc_r - acc_b
winner = "🏆 ROBUST WIN" if diff > 0 else "Baseline Win"
df = pd.DataFrame(columns=['Corruption', 'Baseline Accuracy', 'Robust Accuracy', 'Difference', 'Winner'])
df.loc[0] = ['Clean Data', acc_b, acc_r, diff, winner]

print(f"{'Clean Data':<20} | {acc_b:.2%}     | {acc_r:.2%}     | {diff:+.2%} {winner}")
results = []

for corruption_file in corruptions:
    file_path = os.path.join(DATA_PATH, corruption_file)

    # Check if file exists
    if not os.path.exists(file_path):
        print(f"{corruption_file:<15} | ⚠️ MISSING FILE (Skipping)")
        continue

    # A. Load Data
    # CIFAR-10-C is (N, 32, 32, 3) usually 0-255 uint8
    x_data = np.load(file_path)

    # B. Preprocess (Normalize 0-1)
    # This is critical. If we don't divide by 255, accuracy will be 10%.
    x_data = x_data.astype('float32') / 255.0

    # C. Evaluate Baseline (Silence the logs to prevent crash)
    loss_b, acc_b = model_baseline.evaluate(x_data, y_test_corrupt, batch_size=128, verbose=0)

    # D. Evaluate Robust
    loss_r, acc_r = model_robust.evaluate(x_data, y_test_corrupt, batch_size=128, verbose=0)

    # E. Compare
    diff = acc_r - acc_b
    winner = "🏆 ROBUST WIN" if diff > 0 else "Baseline Win"

    # Print live result
    print(f"{corruption_file[:-4]:<20} | {acc_b:.2%}     | {acc_r:.2%}     | {diff:+.2%} {winner}")

    # Append Results to Dataframe for exporting
    df.loc[len(df)] = [corruption_file[:-4], acc_b, acc_r, diff, winner]
    results.append((corruption_file, acc_b, acc_r, diff))

print("="*72)
df.to_csv('corruptions_results.csv', index=False)

In [ ]:
#Convert the one hot encoded predictions to numerical singular column for ease of interpretation

y_preds_robust = np.argmax(y_preds_robust, axis=1)
y_preds_baseline = np.argmax(y_preds_baseline, axis=1)
y_test_uncode = np.argmax(y_test, axis=1)

In [ ]:
# Print first 10 records of the predictions for comparison and verification of formatting.
print(y_preds_robust[:10])
print(y_test_uncode[:10])
print(y_preds_baseline[:10])


In [ ]:
# Generate a Pandas Dataframe for exporting tabular data to CSV
final_predictions = np.column_stack((y_test_uncode, y_preds_baseline, y_preds_robust))
labels = ['Y Test', 'Baseline Predictions', 'Robust Predictions']
df = pd.DataFrame(final_predictions, columns=labels)

In [ ]:
# Export Datafram to CSV
df.to_csv('final_predictions.csv', index=True)

In [ ]:
# Save Results and Models to Google Drive
!cp 'final_predictions.csv' '/content/drive/MyDrive/Colab Notebooks/final_predictions.csv'
!cp 'corruptions_results.csv' '/content/drive/MyDrive/Colab Notebooks/corruptions_results.csv'
!cp 'final_baseline.keras' '/content/drive/MyDrive/Colab Notebooks/final_baseline.keras'
!cp 'final_robust.keras' '/content/drive/MyDrive/Colab Notebooks/final_robust.keras'

In [ ]:
# Generate first 25 images with true values and predicted values on clean
plt.figure(figsize=(25, 25))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[i], cmap=plt.cm.binary)
    plt.xlabel(f"True: {y_test_uncode[i]} | Baseline: {y_preds_baseline[i]} | Robust: {y_preds_robust[i]}")
plt.show()

In [ ]:
x_noise = np.load(os.path.join(DATA_PATH, 'gaussian_noise.npy'))
y_noise = np.load(os.path.join(DATA_PATH, 'labels.npy'))
y_preds_baseline_noise = model_baseline.predict(x_noise)
y_preds_robust_noise = model_robust.predict(x_noise)
y_preds_baseline_noise = np.argmax(y_preds_baseline_noise, axis=1)
y_preds_robust_noise = np.argmax(y_preds_robust_noise, axis=1)
plt.figure(figsize=(25, 25))
for i in range(50):
    plt.subplot(10, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_noise[i], cmap=plt.cm.binary)
    plt.xlabel(f"True: {y_noise[i]} | Baseline: {y_preds_baseline_noise[i]} | Robust: {y_preds_robust_noise[i]}")
    plt.tight_layout()
plt.show()

In [ ]:
# Set up the plot style
plt.style.use('ggplot')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# A. Loss Curves
ax1.plot(history_robust.history['loss'], label='Training Loss (Robust)', linewidth=2)
ax1.plot(history_robust.history['val_loss'], label='Validation Loss (Robust)', linestyle='--', linewidth=2)
ax1.plot(history_baseline.history['loss'], label='Training Loss (Baseline)', linewidth=2)
ax1.plot(history_baseline.history['val_loss'], label='Validation Loss (Baseline)', linestyle='--', linewidth=2)
ax1.set_title('Loss Curves (Categorical Crossentropy)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# B. Accuracy Curves
ax2.plot(history_robust.history['accuracy'], label='Training Accuracy (Robust)', linewidth=2)
ax2.plot(history_robust.history['val_accuracy'], label='Validation Accuracy (Robust)', linestyle='--', linewidth=2)
ax2.plot(history_baseline.history['accuracy'], label='Training Accuracy (Baseline)', linewidth=2)
ax2.plot(history_baseline.history['val_accuracy'], label='Validation Accuracy (Baseline)', linestyle='--', linewidth=2)
ax2.set_title('Accuracy Curves')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# C. Confusion Matrix
print("\nGenerating Confusion Matrix...")
# Get predictions
y_pred_probs_robust = model_robust.predict(x_test)
y_pred_robust = np.argmax(y_pred_probs_robust, axis=1)
y_true = y_test.argmax(axis=1)
y_pred_probs_baseline = model_baseline.predict(x_test)
y_pred_baseline = np.argmax(y_pred_probs_baseline, axis=1)


# Calculate Matrix
cm = confusion_matrix(y_true, y_pred_robust)
cm2 = confusion_matrix(y_true, y_pred_baseline)

# Plot Heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title('Confusion Matrix: Baseline Model')
ax2.set_title('Confusion Matrix: Robust Model')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')
ax2.set_ylabel('True Label')
ax2.set_xlabel('Predicted Label')
plt.show()

In [ ]:
# --- 6. FINAL METRICS REPORT ---
loss, acc = model_robust.evaluate(x_test, y_test, verbose=0)
loss2, acc2 = model_baseline.evaluate(x_test, y_test, verbose=0)
print("-" * 30)
print(f"BASELINE ACCURACY: {acc2*100:.2f}%")
print(f"BASELINE LOSS:     {loss2:.4f}")
print("-" * 30)
print(f"ROBUST ACCURACY:   {acc*100:.2f}%")
print(f"ROBUST LOSS:       {loss:.4f}")
print("-" * 30)